In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
file_path = "/content/drive/MyDrive/crop_health_jammu/data/WP2_FINAL_PCA_CHS_2017_2025.csv"
df = pd.read_csv(file_path)

print("Shape:", df.shape)
df.head()

Shape: (5320, 13)


,Year,Month,point_id,NDVI,NDMI,NDWI,EVI,LST,LST_anomaly,Rainfall,Rainfall_lag1,CHS_raw,CHS
0,2019,6,0,0.743909,0.152572,-0.671779,0.480969,32.134938,2.393026,62.198192,73.934509,1.059320,0.722705
1,2019,7,0,0.518622,0.161243,-0.404965,0.481283,24.086269,-1.333929,394.833817,62.198192,0.127524,0.629787
2,2019,8,0,0.731209,0.253154,-0.625645,0.599984,24.796311,0.696177,253.643950,394.833817,1.850118,0.801562
3,2019,9,0,0.772564,0.254707,-0.652993,0.471784,23.362406,-0.929274,205.866142,253.643950,1.648132,0.781421
4,2020,6,0,0.753094,0.116774,-0.654536,0.470853,28.933779,-0.808132,78.843528,205.866142,1.022815,0.719064


In [ ]:
print("Year range:", df['Year'].min(), "-", df['Year'].max())
print("Missing CHS:", df['CHS'].isna().sum())

print(df.groupby(['Year','Month']).size().head(15))

Year range: 2017 - 2025
Missing CHS: 0
Year  Month
2017  9          1
2018  6         21
2019  6         81
      7        200
      8        198
      9        200
2020  6        199
      7        200
      8        196
      9        200
2021  6        200
      7        200
      8        196
      9        199
2022  6        200
dtype: int64


In [ ]:
df = df.sort_values(['point_id','Year','Month']).reset_index(drop=True)

In [ ]:
df

,Year,Month,point_id,NDVI,NDMI,NDWI,EVI,LST,LST_anomaly,Rainfall,Rainfall_lag1,CHS_raw,CHS
0,2019,6,0,0.743909,0.152572,-0.671779,0.480969,32.134938,2.393026,62.198192,73.934509,1.059320,0.722705
1,2019,7,0,0.518622,0.161243,-0.404965,0.481283,24.086269,-1.333929,394.833817,62.198192,0.127524,0.629787
2,2019,8,0,0.731209,0.253154,-0.625645,0.599984,24.796311,0.696177,253.643950,394.833817,1.850118,0.801562
3,2019,9,0,0.772564,0.254707,-0.652993,0.471784,23.362406,-0.929274,205.866142,253.643950,1.648132,0.781421
4,2020,6,0,0.753094,0.116774,-0.654536,0.470853,28.933779,-0.808132,78.843528,205.866142,1.022815,0.719064
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5315,2024,8,199,0.357069,0.101340,-0.486716,0.128500,16.267047,-7.833086,270.897670,259.546626,-0.810192,0.536279
5316,2024,9,199,0.275596,0.189278,-0.292706,0.200945,17.908747,-6.382934,180.906539,270.897670,-1.025437,0.514815
5317,2025,6,199,-0.042654,0.282751,0.064815,0.439898,15.665724,-14.076187,126.261251,180.906539,-1.764937,0.441072
5318,2025,8,199,0.018600,0.079376,0.001326,0.197274,16.527382,-7.572752,528.857029,126.261251,-2.658001,0.352017


In [ ]:
train_df = df[df['Year'] <= 2023]
test_df  = df[df['Year'] >= 2024]

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (3824, 13)
Test shape: (1496, 13)


In [ ]:
features = ['NDVI','EVI','NDMI','NDWI','LST','Rainfall','Rainfall_lag1']
target = 'CHS'

def create_sequences(data, window=6):
    X, y = [], []

    for pid in data['point_id'].unique():
        point_data = data[data['point_id'] == pid]
        point_data = point_data.reset_index(drop=True)

        for i in range(window, len(point_data)):
            X.append(point_data[features].iloc[i-window:i].values)
            y.append(point_data[target].iloc[i])

    return np.array(X), np.array(y)

X_train, y_train = create_sequences(train_df, window=6)
X_test, y_test   = create_sequences(test_df, window=6)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (2624, 6, 7)
X_test shape: (296, 6, 7)


In [ ]:
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat  = X_test.reshape(X_test.shape[0], -1)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Flatten sequences
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat  = X_test.reshape(X_test.shape[0], -1)

lr = LinearRegression()
lr.fit(X_train_flat, y_train)

y_pred_lr = lr.predict(X_test_flat)

print("Linear Regression Results")
print("R2:", r2_score(y_test, y_pred_lr))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_lr)))
print("MAE:", mean_absolute_error(y_test, y_pred_lr))

Linear Regression Results
R2: 0.5892998821863085
RMSE: 0.10330302537484373
MAE: 0.07386555623439213


In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=400,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_flat, y_train)
y_pred_rf = rf.predict(X_test_flat)

print("Random Forest Results")
print("R2:", r2_score(y_test, y_pred_rf))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_rf)))
print("MAE:", mean_absolute_error(y_test, y_pred_rf))

Random Forest Results
R2: 0.5994697412305765
RMSE: 0.10201600034631697
MAE: 0.07408771960975913


In [ ]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb.fit(X_train_flat, y_train)
y_pred_xgb = xgb.predict(X_test_flat)

print("XGBoost Results")
print("R2:", r2_score(y_test, y_pred_xgb))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_xgb)))
print("MAE:", mean_absolute_error(y_test, y_pred_xgb))

XGBoost Results
R2: 0.5794185121324913
RMSE: 0.10453836501808546
MAE: 0.0753309471265938


In [ ]:
from sklearn.preprocessing import StandardScaler

# We scale per feature across entire training set
scaler = StandardScaler()

# Reshape to 2D for scaling
X_train_2d = X_train.reshape(-1, X_train.shape[2])
X_test_2d  = X_test.reshape(-1, X_test.shape[2])

# Fit only on training data
scaler.fit(X_train_2d)

X_train_scaled = scaler.transform(X_train_2d).reshape(X_train.shape)
X_test_scaled  = scaler.transform(X_test_2d).reshape(X_test.shape)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

model = Sequential([
    LSTM(64, input_shape=(6, 7), return_sequences=False),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
])

model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train_scaled,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/100


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


66/66 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - loss: 0.2172 - mae: 0.3865 - val_loss: 0.0267 - val_mae: 0.1334
Epoch 2/100
66/66 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0279 - mae: 0.1315 - val_loss: 0.0173 - val_mae: 0.1020
Epoch 3/100
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0216 - mae: 0.1154 - val_loss: 0.0159 - val_mae: 0.0980
Epoch 4/100
66/66 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0191 - mae: 0.1070 - val_loss: 0.0172 - val_mae: 0.1006
Epoch 5/100
66/66 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0203 - mae: 0.1119 - val_loss: 0.0155 - val_mae: 0.0968
Epoch 6/100
66/66 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0189 - mae: 0.1070 - val_loss: 0.0150 - val_mae: 0.0943
Epoch 7/100
66/66 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0180 - mae: 0.1056 - val_loss: 0.0146 - val_mae: 0.0932
Epoch 8/100
66/66 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0166 - mae: 0.1007 - val_loss: 0.0154 - val_mae: 0.0997
Epoch 9/100
66/66 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0169 - mae: 0.

In [ ]:
y_pred_lstm = model.predict(X_test_scaled).flatten()

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

print("LSTM Results")
print("R2:", r2_score(y_test, y_pred_lstm))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_lstm)))
print("MAE:", mean_absolute_error(y_test, y_pred_lstm))

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
LSTM Results
R2: 0.4762042238555996
RMSE: 0.11666259242554387
MAE: 0.08616595994108633


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor

rf_base = RandomForestRegressor(random_state=42, n_jobs=-1)

param_grid = {
    'n_estimators': [300, 500, 700, 1000],
    'max_depth': [None, 10, 15, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]
}

rf_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=param_grid,
    n_iter=25,              # not too large
    cv=3,                   # time-series already structured
    scoring='r2',
    verbose=2,
    random_state=42,
    n_jobs=-1
)

rf_search.fit(X_train_flat, y_train)

print("Best Parameters:")
print(rf_search.best_params_)

Fitting 3 folds for each of 25 candidates, totalling 75 fits
Best Parameters:
{'n_estimators': 500, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': 15}


In [ ]:
best_rf = rf_search.best_estimator_

y_pred_rf_tuned = best_rf.predict(X_test_flat)

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

print("Tuned Random Forest Results")
print("R2:", r2_score(y_test, y_pred_rf_tuned))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_rf_tuned)))
print("MAE:", mean_absolute_error(y_test, y_pred_rf_tuned))

Tuned Random Forest Results
R2: 0.6078226243143723
RMSE: 0.10094664624594421
MAE: 0.07226643336924696


In [ ]:
import pickle
import os

os.makedirs("/content/drive/MyDrive/crop_health_jammu/models", exist_ok=True)

model_path = "/content/drive/MyDrive/crop_health_jammu/models/basic_rf_model.pkl"

with open(model_path, "wb") as f:
    pickle.dump(rf, f)

print("Model saved successfully.")

Model saved successfully.


In [ ]:
np.save("/content/drive/MyDrive/crop_health_jammu/models/X_test_flat.npy", X_test_flat)
np.save("/content/drive/MyDrive/crop_health_jammu/models/y_test.npy", y_test)